In [1]:
import os
import random
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import gc 
import time
import copy
import math
from torchvision.transforms import RandAugment

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchinfo
from torchinfo import summary
from torch.utils.data import Dataset, DataLoader
from torch.utils.data import WeightedRandomSampler
from torchvision import transforms
from torchvision import models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report, accuracy_score, f1_score, cohen_kappa_score, roc_auc_score, precision_score, recall_score
from PIL import Image


import hashlib
import shutil
import ssl
import tempfile
import warnings
from pathlib import Path
from urllib.error import URLError
from urllib.parse import urlparse
from urllib.request import HTTPSHandler, Request, build_opener

import torch
import torch.nn as nn
from torch.hub import get_dir as get_torch_hub_dir
from torchvision import models
from torchvision.models import VGG16_Weights, vgg16

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

In [2]:
LC25000_PATH = os.path.expanduser(
    "~/Documentos/lung_baseline/LC25000/"
    "lung_colon_image_set/lung_image_sets"
)

BATCH_SIZE = 32
NUM_WORKERS = 4
random_seed = 42

def seed_everything(seed):

    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Dispositivo:", device)
print("LC25000:", LC25000_PATH)

class_names = ["ACA", "SCC", "NOR"]

class_to_idx = {
    "lung_aca": 0,
    "lung_scc": 1,
    "lung_n": 2
}

print("Classes:", class_to_idx, class_names)

Dispositivo: cuda
LC25000: /home/joy/Documentos/lung_baseline/LC25000/lung_colon_image_set/lung_image_sets
Classes: {'lung_aca': 0, 'lung_scc': 1, 'lung_n': 2} ['ACA', 'SCC', 'NOR']


In [3]:
def create_model(model_name, num_classes=3, feature_extract=True):
    model = None

    if model_name == 'resnet50':
        weights = models.ResNet50_Weights.DEFAULT
        model = models.resnet50(weights=weights)

        if feature_extract:
            for param in model.parameters():
                param.requires_grad = False

        num_ftrs = model.fc.in_features
        model.fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_ftrs, num_classes)
        )

    elif model_name == 'efficientnet_b0':
        weights = models.EfficientNet_B0_Weights.DEFAULT
        model = models.efficientnet_b0(weights=weights)

        if feature_extract:
            for param in model.parameters():
                param.requires_grad = False

        num_ftrs = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_ftrs, num_classes)
        )

    elif model_name == 'efficientnet_b3':
        weights = models.EfficientNet_B3_Weights.DEFAULT
        model = models.efficientnet_b3(weights=weights)

        if feature_extract:
            for param in model.parameters():
                param.requires_grad = False

        num_ftrs = model.classifier[1].in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_ftrs, num_classes)
        )

    elif model_name == 'densenet121':
        weights = models.DenseNet121_Weights.DEFAULT
        model = models.densenet121(weights=weights)

        if feature_extract:
            for param in model.parameters():
                param.requires_grad = False

        num_ftrs = model.classifier.in_features
        model.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_ftrs, num_classes)
        )

    elif model_name == 'convnext_tiny':

        weights = models.ConvNeXt_Tiny_Weights.DEFAULT
        model = models.convnext_tiny(weights=weights)

        if feature_extract:
            for param in model.parameters():
                param.requires_grad = False

       
        num_ftrs = model.classifier[2].in_features

        model.classifier[2] = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_ftrs, num_classes)
        )

        # libera apenas o classifier
        for param in model.classifier.parameters():
            param.requires_grad = True

    elif model_name == 'vit_b16':

        weights = models.ViT_B_16_Weights.DEFAULT
        model = models.vit_b_16(weights=weights)

        # congela tudo
        for param in model.parameters():
            param.requires_grad = False

        # libera últimas camadas do encoder
        for param in model.encoder.layers[-4:].parameters():
            param.requires_grad = True

        # cabeça
        num_ftrs = model.heads.head.in_features

        model.heads.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_ftrs, num_classes)
        )

        for param in model.heads.parameters():
            param.requires_grad = True
    return model.to(device)

In [4]:
def load_model_weights(model, path, device):
    
    checkpoint = torch.load(
        path,
        map_location=device
    )

    # Caso o arquivo tenha sido salvo como state_dict diretamente
    if isinstance(checkpoint, dict):

        if "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]

        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]

        else:
            state_dict = checkpoint

    else:
        state_dict = checkpoint

    # Remove "module." caso tenha sido salvo com DataParallel
    state_dict = {
        k.replace("module.", "", 1): v
        for k, v in state_dict.items()
    }

    model.load_state_dict(state_dict)

    model.to(device)
    model.eval()

    return model

In [5]:
def load_all_models(path):

    models_20x = {}
    models_40x = {}

    model_names = [
        "resnet50",
        "efficientnet_b3",
        "densenet121",
        "vit_b16",
        "convnext_tiny"
    ]

    for model_name in model_names:

        # -------------------------
        # 20x
        # -------------------------
        file_20x = os.path.join(
            path,
            f"best_Rand_{model_name}_20x.pth"
        )

        if os.path.exists(file_20x):

            model = create_model(
                model_name,
                num_classes=3
            )

            model = load_model_weights(
                model,
                file_20x,
                device
            )

            models_20x[
                f"best_Rand_{model_name}_20x"
            ] = model

            print(f"✓ Carregado: {os.path.basename(file_20x)}")

        else:
            print(f"⚠️ Não encontrado: {file_20x}")


        # -------------------------
        # 40x
        # -------------------------
        file_40x = os.path.join(
            path,
            f"best_Rand_{model_name}_40x.pth"
        )

        if os.path.exists(file_40x):

            model = create_model(
                model_name,
                num_classes=3
            )

            model = load_model_weights(
                model,
                file_40x,
                device
            )

            models_40x[
                f"best_Rand_{model_name}_40x"
            ] = model

            print(f"✓ Carregado: {os.path.basename(file_40x)}")

        else:
            print(f"⚠️ Não encontrado: {file_40x}")

    return models_20x, models_40x

In [6]:
path = os.path.expanduser(
    "~/Documentos/lung_baseline/Best_models_Rand"
)

models_20x, models_40x = load_all_models(path)


models_list_20x = [
    models_20x["best_Rand_resnet50_20x"],
    models_20x["best_Rand_efficientnet_b3_20x"],
    models_20x["best_Rand_densenet121_20x"],
    models_20x["best_Rand_vit_b16_20x"],
    models_20x["best_Rand_convnext_tiny_20x"],
]


models_list_40x = [
    models_40x["best_Rand_resnet50_40x"],
    models_40x["best_Rand_efficientnet_b3_40x"],
    models_40x["best_Rand_densenet121_40x"],
    models_40x["best_Rand_vit_b16_40x"],
    models_40x["best_Rand_convnext_tiny_40x"],
]


model_names = [
    "ResNet50",
    "EfficientNet-B3",
    "DenseNet121",
    "ViT-B16",
    "ConvNeXt-Tiny"
]


print("\nModelos carregados:")
print("20x:", len(models_list_20x))
print("40x:", len(models_list_40x))

print("\nModelos:")
for name in model_names:
    print("-", name)

✓ Carregado: best_Rand_resnet50_20x.pth
✓ Carregado: best_Rand_resnet50_40x.pth
✓ Carregado: best_Rand_efficientnet_b3_20x.pth
✓ Carregado: best_Rand_efficientnet_b3_40x.pth
✓ Carregado: best_Rand_densenet121_20x.pth
✓ Carregado: best_Rand_densenet121_40x.pth
✓ Carregado: best_Rand_vit_b16_20x.pth
✓ Carregado: best_Rand_vit_b16_40x.pth
✓ Carregado: best_Rand_convnext_tiny_20x.pth
✓ Carregado: best_Rand_convnext_tiny_40x.pth

Modelos carregados:
20x: 5
40x: 5

Modelos:
- ResNet50
- EfficientNet-B3
- DenseNet121
- ViT-B16
- ConvNeXt-Tiny


In [7]:
external_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [8]:
class LC25000Dataset(Dataset):

    def __init__(self, root_dir, transform=None):

        self.root_dir = root_dir
        self.transform = transform

        self.samples = []

        for folder, label in class_to_idx.items():

            class_dir = os.path.join(root_dir, folder)

            if not os.path.isdir(class_dir):
                print(f"AVISO: pasta não encontrada: {class_dir}")
                continue

            for filename in sorted(os.listdir(class_dir)):

                if filename.lower().endswith(
                    (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")
                ):

                    image_path = os.path.join(
                        class_dir,
                        filename
                    )

                    self.samples.append(
                        (image_path, label)
                    )

        print("\nDataset LC25000 criado")
        print("Total de imagens:", len(self.samples))

        for label, name in enumerate(class_names):

            count = sum(
                1 for _, y in self.samples
                if y == label
            )

            print(f"{name}: {count}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        image_path, label = self.samples[idx]

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


external_dataset = LC25000Dataset(
    root_dir=LC25000_PATH,
    transform=external_transform
)

external_loader = DataLoader(
    external_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=False
)

print("\nDataLoader criado com sucesso!")


Dataset LC25000 criado
Total de imagens: 15000
ACA: 5000
SCC: 5000
NOR: 5000

DataLoader criado com sucesso!


In [9]:
images, labels = next(iter(external_loader))

print("Shape das imagens:", images.shape)
print("Shape dos labels:", labels.shape)
print("Labels:", labels[:10])

print("\nEsperado:")
print("Imagens: [batch, 3, 224, 224]")
print("Classes: 0=ACA, 1=SCC, 2=NOR")

Shape das imagens: torch.Size([32, 3, 224, 224])
Shape dos labels: torch.Size([32])
Labels: tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

Esperado:
Imagens: [batch, 3, 224, 224]
Classes: 0=ACA, 1=SCC, 2=NOR


In [10]:
def evaluate_external_model(
    model,
    loader,
    model_name
):

    model.eval()

    all_labels = []
    all_preds = []
    all_probs = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(device)

            outputs = model(images)

            probs = F.softmax(outputs, dim=1)

            preds = torch.argmax(
                probs,
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy()
            )

            all_preds.extend(
                preds.cpu().numpy()
            )

            all_probs.extend(
                probs.cpu().numpy()
            )

    y_true = np.array(all_labels)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)


    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    precision = precision_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    kappa = cohen_kappa_score(
        y_true,
        y_pred
    )

    try:

        auc = roc_auc_score(
            y_true,
            y_prob,
            multi_class="ovr",
            average="macro"
        )

    except Exception:

        auc = np.nan

    # ========================================================
    # RESULTADO
    # ========================================================

    print("\n" + "=" * 70)
    print(f"VALIDAÇÃO EXTERNA — {model_name}")
    print("=" * 70)

    print(
        f"Accuracy : {accuracy:.4f}"
    )

    print(
        f"ROC-AUC  : {auc:.4f}"
    )

    print(
        f"Kappa    : {kappa:.4f}"
    )

    print(
        f"Precision: {precision:.4f}"
    )

    print(
        f"Recall   : {recall:.4f}"
    )

    print(
        f"F1-score : {f1:.4f}"
    )

    print("\nRELATÓRIO DE CLASSIFICAÇÃO:")

    print(
        classification_report(
            y_true,
            y_pred,
            target_names=class_names,
            digits=4,
            zero_division=0
        )
    )

    # ========================================================
    # MATRIZ DE CONFUSÃO
    # ========================================================

    cm = confusion_matrix(
        y_true,
        y_pred
    )

    print("MATRIZ DE CONFUSÃO:")
    print(cm)

    # ========================================================
    # RETORNO
    # ========================================================

    return {
        "model": model_name,
        "accuracy": accuracy,
        "roc_auc": auc,
        "kappa": kappa,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "confusion_matrix": cm
    }

In [11]:
results_external_20x = []

for model, name in zip(
    models_list_20x,
    model_names
):

    model = model.to(device)

    result = evaluate_external_model(
        model=model,
        loader=external_loader,
        model_name=f"{name} — 20x"
    )

    results_external_20x.append(result)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


VALIDAÇÃO EXTERNA — ResNet50 — 20x
Accuracy : 0.5961
ROC-AUC  : 0.8122
Kappa    : 0.3941
Precision: 0.6580
Recall   : 0.5961
F1-score : 0.5062

RELATÓRIO DE CLASSIFICAÇÃO:
              precision    recall  f1-score   support

         ACA     0.5612    0.0156    0.0304      5000
         SCC     0.4597    0.9878    0.6274      5000
         NOR     0.9531    0.7848    0.8608      5000

    accuracy                         0.5961     15000
   macro avg     0.6580    0.5961    0.5062     15000
weighted avg     0.6580    0.5961    0.5062     15000

MATRIZ DE CONFUSÃO:
[[  78 4729  193]
 [  61 4939    0]
 [   0 1076 3924]]

VALIDAÇÃO EXTERNA — EfficientNet-B3 — 20x
Accuracy : 0.6573
ROC-AUC  : 0.8324
Kappa    : 0.4860
Precision: 0.6283
Recall   : 0.6573
F1-score : 0.5761

RELATÓRIO DE CLASSIFICAÇÃO:
              precision    recall  f1-score   support

         ACA     0.4496    0.0660    0.1151      5000
         SCC     0.5161    0.9176    0.6606      5000
         NOR     0.9193    0

In [12]:
results_external_40x = []

for model, name in zip(
    models_list_40x,
    model_names
):

    model = model.to(device)

    result = evaluate_external_model(
        model=model,
        loader=external_loader,
        model_name=f"{name} — 40x"
    )

    results_external_40x.append(result)

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


VALIDAÇÃO EXTERNA — ResNet50 — 40x
Accuracy : 0.6065
ROC-AUC  : 0.8718
Kappa    : 0.4098
Precision: 0.7446
Recall   : 0.6065
F1-score : 0.5841

RELATÓRIO DE CLASSIFICAÇÃO:
              precision    recall  f1-score   support

         ACA     0.4751    0.9338    0.6298      5000
         SCC     0.9307    0.2658    0.4135      5000
         NOR     0.8280    0.6200    0.7091      5000

    accuracy                         0.6065     15000
   macro avg     0.7446    0.6065    0.5841     15000
weighted avg     0.7446    0.6065    0.5841     15000

MATRIZ DE CONFUSÃO:
[[4669   99  232]
 [3259 1329  412]
 [1900    0 3100]]

VALIDAÇÃO EXTERNA — EfficientNet-B3 — 40x
Accuracy : 0.6788
ROC-AUC  : 0.9091
Kappa    : 0.5182
Precision: 0.7388
Recall   : 0.6788
F1-score : 0.6525

RELATÓRIO DE CLASSIFICAÇÃO:
              precision    recall  f1-score   support

         ACA     0.5747    0.7032    0.6325      5000
         SCC     0.9405    0.3448    0.5046      5000
         NOR     0.7011    0

In [13]:
def results_to_dataframe(results):

    rows = []

    for r in results:

        rows.append({
            "Modelo": r["model"],
            "Accuracy": r["accuracy"],
            "ROC-AUC": r["roc_auc"],
            "Kappa": r["kappa"],
            "Precision": r["precision"],
            "Recall": r["recall"],
            "F1-score": r["f1"]
        })

    return pd.DataFrame(rows)


df_external_20x = results_to_dataframe(
    results_external_20x
)

df_external_40x = results_to_dataframe(
    results_external_40x
)


print("\n==================== 20X ====================")
display(
    df_external_20x.round(4)
)

print("\n==================== 40X ====================")
display(
    df_external_40x.round(4)
)


==================== 20X ====================


,Modelo,Accuracy,ROC-AUC,Kappa,Precision,Recall,F1-score
0,ResNet50 — 20x,0.5961,0.8122,0.3941,0.6580,0.5961,0.5062
1,EfficientNet-B3 — 20x,0.6573,0.8324,0.4860,0.6283,0.6573,0.5761
2,DenseNet121 — 20x,0.6505,0.8309,0.4758,0.8286,0.6505,0.5436
3,ViT-B16 — 20x,0.7121,0.8838,0.5681,0.7389,0.7121,0.6549
4,ConvNeXt-Tiny — 20x,0.7668,0.9389,0.6502,0.8282,0.7668,0.7391



==================== 40X ====================


,Modelo,Accuracy,ROC-AUC,Kappa,Precision,Recall,F1-score
0,ResNet50 — 40x,0.6065,0.8718,0.4098,0.7446,0.6065,0.5841
1,EfficientNet-B3 — 40x,0.6788,0.9091,0.5182,0.7388,0.6788,0.6525
2,DenseNet121 — 40x,0.7011,0.9264,0.5517,0.7735,0.7011,0.6918
3,ViT-B16 — 40x,0.6547,0.9261,0.4820,0.7598,0.6547,0.5768
4,ConvNeXt-Tiny — 40x,0.6795,0.9201,0.5193,0.7909,0.6795,0.6060
